In [14]:

# ShopWave production agent — dependencies (langchain-core 1.x for LangGraph 0.6+)
%pip install -q "langgraph>=0.2" "langchain-core>=1.2.2,<3" "langchain-groq>=1.0.0" python-dotenv



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Data loading

Same JSON sources as before (`data/` under repo root).


In [15]:

import json
import re
import threading
import time
import random
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any, Callable, List

# Repo root: cwd or parent when running from notebooks/
_root = Path.cwd()
REPO_ROOT = _root if (_root / "data").is_dir() else _root.parent
DATA_DIR = REPO_ROOT / "data"
AUDIT_LOG_DIR = REPO_ROOT / "audit_logs"
AUDIT_LOG_PATH = REPO_ROOT / "audit_log.json"  # legacy single file; per-ticket logs use AUDIT_LOG_DIR


def _env_float(name: str, default: str) -> float:
    # Strip inline # comments — raw getenv may include them if .env has trailing comments
    return float(__import__("os").getenv(name, default).split("#", 1)[0].strip())


CONFIDENCE_THRESHOLD = _env_float("SHOPWAVE_CONFIDENCE_THRESHOLD", "0.6")
FAILURE_SIM_RATE = _env_float("SHOPWAVE_TOOL_FAILURE_RATE", "0.12")
MAX_TOOL_RETRIES = 3

with open(DATA_DIR / "orders.json", encoding="utf-8") as f:
    ORDERS = {o["order_id"]: o for o in json.load(f)}
with open(DATA_DIR / "customers.json", encoding="utf-8") as f:
    CUSTOMERS_BY_ID = {c["customer_id"]: c for c in json.load(f)}
    CUSTOMERS_BY_EMAIL = {c["email"].lower(): c for c in CUSTOMERS_BY_ID.values()}
with open(DATA_DIR / "products.json", encoding="utf-8") as f:
    PRODUCTS = {p["product_id"]: p for p in json.load(f)}
with open(DATA_DIR / "tickets.json", encoding="utf-8") as f:
    TICKETS = {t["ticket_id"]: t for t in json.load(f)}

KNOWLEDGE_BASE = (DATA_DIR / "knowledge-base.md").read_text(encoding="utf-8")


def _parse_date(s: str) -> date:
    return datetime.fromisoformat(s.replace("Z", "+00:00")).date() if "T" in s else date.fromisoformat(s[:10])


ORDER_ID_RE = re.compile(r"\b(ORD-\d+)\b", re.I)


def extract_order_id(text: str) -> str | None:
    m = ORDER_ID_RE.search(text or "")
    return m.group(1).upper() if m else None


def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def as_of_date_from_state(state: dict) -> date:
    s = state.get("as_of") or ""
    if len(s) >= 10:
        try:
            return date.fromisoformat(s[:10])
        except ValueError:
            pass
    return date.today()


def ticket_context(ticket_id: str) -> dict:
    t = TICKETS.get(ticket_id)
    if not t:
        return {"ticket_id": ticket_id, "ticket": "", "as_of": "", "customer_email": "", "tier": None}
    body = f"Subject: {t['subject']}\n\n{t['body']}"
    return {
        "ticket_id": ticket_id,
        "ticket": body,
        "as_of": (t.get("created_at") or "")[:10],
        "customer_email": t.get("customer_email", ""),
        "tier": t.get("tier"),
    }


def latest_order_id_for_email(email: str) -> str | None:
    """Resolve order when the ticket body has no ORD-* (e.g. lookup by email → customer → orders)."""
    c = CUSTOMERS_BY_EMAIL.get((email or "").strip().lower())
    if not c:
        return None
    cid = c["customer_id"]
    rows = [o for o in ORDERS.values() if o.get("customer_id") == cid]
    if not rows:
        return None
    rows.sort(key=lambda o: str(o.get("order_date") or ""), reverse=True)
    return rows[0]["order_id"]


## 2. Logging — per-ticket audit (`audit_logs/<ticket_id>.json`)

Each run: steps with thought, action, input, output, timestamp. Escalations append to `audit_logs/escalations.json`. Thread-safe.


In [16]:

_audit_lock = threading.Lock()


def audit_path_for_ticket(ticket_id: str) -> Path:
    safe = re.sub(r"[^\w\-]+", "_", str(ticket_id).strip()) or "unknown"
    return AUDIT_LOG_DIR / f"{safe}.json"


def _normalize_ticket_audit_file(raw: object, ticket_id: str) -> dict:
    """Single-ticket JSON: { version, ticket_id, runs: [...] }."""
    if raw is None or (isinstance(raw, dict) and not raw):
        return {"version": 1, "ticket_id": ticket_id, "runs": []}
    if isinstance(raw, list):
        return {"version": 1, "ticket_id": ticket_id, "runs": list(raw)}
    if not isinstance(raw, dict):
        return {"version": 1, "ticket_id": ticket_id, "runs": []}
    out = dict(raw)
    out.setdefault("version", 1)
    out["ticket_id"] = raw.get("ticket_id", ticket_id)
    out.setdefault("runs", [])
    return out


def _load_ticket_audit_file(ticket_id: str) -> dict:
    path = audit_path_for_ticket(ticket_id)
    if not path.is_file():
        return {"version": 1, "ticket_id": ticket_id, "runs": []}
    try:
        raw = json.loads(path.read_text(encoding="utf-8"))
        return _normalize_ticket_audit_file(raw, ticket_id)
    except json.JSONDecodeError:
        return {"version": 1, "ticket_id": ticket_id, "runs": [], "corrupted_previous": True}


def _save_ticket_audit_file(ticket_id: str, data: dict) -> None:
    path = audit_path_for_ticket(ticket_id)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)


def append_audit_step(ticket_id: str, run_id: str, step: dict) -> None:
    step = {**step, "timestamp": step.get("timestamp") or _now_iso()}
    with _audit_lock:
        data = _load_ticket_audit_file(ticket_id)
        runs = data.setdefault("runs", [])
        run = next((r for r in runs if r.get("run_id") == run_id), None)
        if run is None:
            run = {"run_id": run_id, "started_at": _now_iso(), "steps": []}
            runs.append(run)
        run.setdefault("steps", []).append(step)
        _save_ticket_audit_file(ticket_id, data)


def finalize_ticket_audit(
    ticket_id: str,
    run_id: str,
    *,
    final_decision: str,
    actions_taken: list,
    escalated: bool,
    extra: dict | None = None,
) -> None:
    with _audit_lock:
        data = _load_ticket_audit_file(ticket_id)
        for run in data.get("runs", []):
            if run.get("run_id") == run_id:
                run["final_decision"] = final_decision
                run["actions_taken"] = actions_taken
                run["escalated"] = escalated
                run["finished_at"] = _now_iso()
                if extra:
                    run["extra"] = extra
                break
        _save_ticket_audit_file(ticket_id, data)


## 3. Retries — simulated failures + exponential backoff


In [17]:

class ToolTimeout(Exception):
    pass  # simulated network timeout


class ToolMalformedResponse(Exception):
    pass  # simulated bad payload


class ToolRetriesExhausted(Exception):
    def __init__(self, cause: Exception):
        super().__init__(str(cause))
        self.cause = cause


def _simulate_tool_chaos() -> None:
    if random.random() > FAILURE_SIM_RATE:
        return
    r = random.random()
    if r < 0.45:
        time.sleep(0.08 + random.random() * 0.05)
        raise ToolTimeout("simulated tool timeout")
    raise ToolMalformedResponse("simulated malformed tool response")


def run_with_retries(
    op_name: str,
    fn: Callable[[], Any],
    *,
    max_retries: int = MAX_TOOL_RETRIES,
    base_delay_s: float = 0.15,
) -> Any:
    last_err: Exception | None = None
    for attempt in range(max_retries + 1):
        try:
            _simulate_tool_chaos()
            out = fn()
            if isinstance(out, dict) and out.get("__malformed__"):
                raise ToolMalformedResponse("handler marked malformed")
            return out
        except (ToolTimeout, ToolMalformedResponse, KeyError, ValueError) as e:
            last_err = e
            if attempt >= max_retries:
                raise ToolRetriesExhausted(e) from e
            delay = base_delay_s * (2**attempt) + random.random() * 0.05
            time.sleep(delay)
    raise ToolRetriesExhausted(last_err)  # pragma: no cover


## 4. Escalation


In [18]:

def escalate(ticket_id: str, summary: str, priority: str) -> dict:
    assert priority in ("low", "medium", "high", "urgent")
    rec = {
        "status": "escalated",
        "ticket_id": ticket_id,
        "priority": priority,
        "summary": summary,
        "created_at": _now_iso(),
    }
    path = AUDIT_LOG_DIR / "escalations.json"
    with _audit_lock:
        AUDIT_LOG_DIR.mkdir(parents=True, exist_ok=True)
        data = {"version": 1, "records": []}
        if path.is_file():
            try:
                data = json.loads(path.read_text(encoding="utf-8"))
            except json.JSONDecodeError:
                pass
        data.setdefault("records", []).append(rec)
        tmp = path.with_suffix(".json.tmp")
        tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
        tmp.replace(path)
    return rec


## 5. Core tools + validation

`classify_ticket` (triage), `search_knowledge_base`, `get_customer`, `get_order`, `get_product`, `check_refund`, `issue_refund`, `send_reply`.


In [19]:

def get_order(order_id: str) -> dict:
    oid = order_id.strip().upper()
    if not re.fullmatch(r"ORD-\d+", oid):
        raise ValueError(f"invalid order_id: {order_id!r}")
    if oid not in ORDERS:
        return {"error": "not_found", "order_id": oid}
    o = ORDERS[oid]
    p = PRODUCTS.get(o["product_id"], {})
    return {
        "order_id": oid,
        "customer_id": o["customer_id"],
        "product_id": o["product_id"],
        "product_name": p.get("name"),
        "status": o["status"],
        "amount": o["amount"],
        "delivery_date": o.get("delivery_date"),
        "return_deadline": o.get("return_deadline"),
        "refund_status": o.get("refund_status"),
        "notes": o.get("notes"),
    }


def get_customer(*, customer_id: str | None = None, email: str | None = None) -> dict:
    if customer_id and customer_id in CUSTOMERS_BY_ID:
        return {"found": True, "customer": CUSTOMERS_BY_ID[customer_id]}
    if email:
        c = CUSTOMERS_BY_EMAIL.get(email.strip().lower())
        if c:
            return {"found": True, "customer": c}
    return {"found": False, "error": "customer not found"}


def check_refund_eligibility(order_id: str, *, as_of: date | None = None) -> dict:
    oid = order_id.strip().upper()
    if not re.fullmatch(r"ORD-\d+", oid):
        raise ValueError(f"invalid order_id: {order_id!r}")
    if oid not in ORDERS:
        return {"eligible": False, "reason": "order not found"}
    o = ORDERS[oid]
    p = PRODUCTS.get(o["product_id"])
    as_of = as_of or date.today()
    rd = o.get("return_deadline")
    if not rd:
        return {"eligible": False, "reason": "no return deadline on order"}
    deadline = _parse_date(rd)
    if as_of > deadline:
        return {
            "eligible": False,
            "reason": f"return window ended on {deadline.isoformat()}",
            "return_deadline": rd,
            "product_notes": (p or {}).get("notes"),
        }
    return {
        "eligible": True,
        "reason": f"within return window (deadline {deadline.isoformat()})",
        "return_deadline": rd,
        "order_status": o["status"],
    }


def issue_refund(order_id: str, amount: float) -> dict:
    oid = order_id.strip().upper()
    if oid not in ORDERS:
        return {"status": "error", "detail": "order not found"}
    return {"status": "success", "order_id": oid, "refunded_amount": amount}


def send_reply(ticket_id: str, message: str) -> dict:
    if not ticket_id or not str(ticket_id).strip():
        raise ValueError("ticket_id required")
    return {"status": "sent", "ticket_id": ticket_id, "message": message}


def classify_ticket_text(
    ticket_text: str, customer_email: str = "", *, ticket_tier: int | None = None
) -> dict:
    """Rule-based urgency, category, and resolvability before resolution tools."""
    text = (ticket_text or "").lower()
    category = "general"
    if re.search(r"\bcancel(?:lation)?\b", text):
        category = "cancellation"
    elif any(
        p in text
        for p in ("wrong size", "wrong colour", "wrong color", "wrong item", "received the wrong", "got the wrong")
    ):
        category = "wrong_item"
    elif "warranty" in text or ("defect" in text and "refund" not in text[:200]):
        category = "warranty"
    elif "refund" in text:
        category = "refund"
    elif "return" in text:
        category = "return"
    elif any(p in text for p in ("where is", "tracking", "hasn't arrived", "not received", "in transit", "where my order")):
        category = "shipping"
    elif any(p in text for p in ("policy", "exchange", "how do i")) and "ord-" not in text:
        category = "general"
    else:
        category = "other"
    urgency = "low"
    if any(w in text for w in ("lawyer", "dispute with my bank", "dispute", "bank")) and "refund" in text:
        urgency = "critical"
    elif any(w in text for w in ("immediately", "urgent", "unacceptable", "right now", "today")):
        urgency = "high"
    elif category in ("wrong_item", "refund", "return", "shipping"):
        urgency = "high" if category in ("wrong_item", "shipping") else "medium"
    if ticket_tier is not None and ticket_tier >= 2 and urgency == "low":
        urgency = "medium"
    has_oid = bool(extract_order_id(ticket_text))
    resolvable = has_oid or bool((customer_email or "").strip())
    return {
        "urgency": urgency,
        "category": category,
        "resolvable": resolvable,
        "signals": {"has_order_id": has_oid, "has_customer_email": bool((customer_email or "").strip())},
        "notes": "rule_based triage",
    }


def search_knowledge_base(query: str, *, top_k: int = 6) -> dict:
    """Callable KB search over `knowledge-base.md` (chunked, token overlap score)."""
    q = (query or "").strip().lower()
    if not q:
        return {"query": query, "matches": [], "match_count": 0}
    tokens = [t for t in re.findall(r"[a-z0-9]+", q) if len(t) > 2][:24]
    chunks = [c.strip() for c in re.split(r"\n-{3,}\s*\n|\n##+\s+", KNOWLEDGE_BASE) if len(c.strip()) > 50]
    if len(chunks) < 4:
        chunks = [c.strip() for c in KNOWLEDGE_BASE.split("\n\n") if len(c.strip()) > 50]
    scored: list[tuple[float, str]] = []
    for ch in chunks:
        low = ch.lower()
        score = sum(2 for t in tokens if t in low) + sum(1 for t in tokens if t in low) * 0.1
        if score > 0:
            scored.append((score, ch[:1600]))
    scored.sort(key=lambda x: -x[0])
    matches = [{"rank": i + 1, "score": round(s, 3), "text": txt} for i, (s, txt) in enumerate(scored[:top_k])]
    return {"query": query, "matches": matches, "match_count": len(matches)}


def get_product(product_id: str) -> dict:
    pid = (product_id or "").strip().upper()
    if not re.fullmatch(r"P\d+", pid):
        return {"error": "invalid_product_id", "product_id": product_id}
    if pid not in PRODUCTS:
        return {"error": "not_found", "product_id": pid}
    p = PRODUCTS[pid]
    return {
        "product_id": pid,
        "name": p.get("name"),
        "category": p.get("category"),
        "price": p.get("price"),
        "warranty_months": p.get("warranty_months"),
        "return_window_days": p.get("return_window_days"),
        "returnable": p.get("returnable"),
        "notes": p.get("notes"),
    }


## 6. Agent state, think / act, LangGraph


In [20]:

from typing import TypedDict


class AgentState(TypedDict, total=False):
    run_id: str
    ticket_id: str
    ticket: str
    as_of: str
    customer_email: str
    order_id: str
    messages: List[str]
    next_step: str
    result: str
    thought: str
    confidence: float
    escalated: bool
    escalation_reason: str
    escalation_priority: str
    actions_taken: List[str]
    audit_chain: List[dict]
    think_cycle: int
    final_decision: str
    tool_failure_escalation: bool
    reply_draft: str
    classification: dict
    ticket_tier: int
    product_id: str


In [21]:

import logging
import os
import re
from dotenv import load_dotenv
from langchain_groq import ChatGroq

for _env in (REPO_ROOT / ".env", _root / ".env"):
    if _env.is_file():
        load_dotenv(_env)
        break


def _make_llm():
    key = os.getenv("GROQ_API_KEY", "").strip()
    if not key:
        return None
    return ChatGroq(api_key=key, model="llama-3.1-8b-instant", temperature=0)


llm = _make_llm()
_LOG = logging.getLogger("shopwave.agent")


def _parse_confidence_from_llm_lines(lines: list[str], default: float = 0.75) -> float:
    """Parse confidence in [0, 1] from LLM lines (handles %, EU commas); avoids unsafe float(whole_line)."""
    if not lines:
        return default
    num_in_01 = re.compile(r"(?<![0-9.])(1(?:\.0+)?|0?\.\d+)(?![0-9.])")
    pct = re.compile(r"(?<!\d)(\d{1,3})\s*%")
    for ln in reversed(lines):
        t = ln.strip().replace(chr(0xA0), " ").replace(",", ".")
        m_pct = pct.search(t.replace(" ", ""))
        if m_pct:
            try:
                v = int(m_pct.group(1))
                if 0 <= v <= 100:
                    x = v / 100.0
                    if 0.0 <= x <= 1.0:
                        return x
            except ValueError:
                pass
        compact = re.sub(r"\s+", "", t)
        for m in num_in_01.finditer(compact):
            try:
                v = float(m.group(1))
                if 0.0 <= v <= 1.0:
                    return v
            except ValueError:
                continue
    return default


def _generate_reply_message(state: dict) -> str:
    """Customer-facing reply from ticket + tool history; LLM when available, else short static fallback."""
    ctx = (state.get("ticket") or "").strip()
    hist = "\n".join(state.get("messages") or [])
    oid = (state.get("order_id") or "").strip().upper()
    if llm is None or os.getenv("SHOPWAVE_OFFLINE"):
        tail = f" Regarding order {oid}, we have applied the outcome from our tools above." if oid else " We have applied the outcome from our tools above."
        return ("Thanks for contacting ShopWave." + tail)[:2000]
    prompt = (
        "You are ShopWave customer support. Write a concise, professional email body in plain text "
        "(max ~1800 characters). Address the customer by context; use tool results below; do not invent "
        "policies beyond the knowledge excerpt. If a refund was issued, note typical processing time; "
        "if not eligible, explain briefly and kindly.\n\n"
        f"Knowledge excerpt:\n{KNOWLEDGE_BASE[:2200]}\n\n"
        f"Order id (if known): {oid or '(none)'}\n\n"
        f"Customer message:\n{ctx}\n\n"
        f"Tool / system log:\n{hist[:5000]}"
    )
    try:
        resp = llm.invoke(prompt)
        body = (getattr(resp, "content", None) or str(resp)).strip()
        if body:
            return body[:2000]
    except Exception as e:
        _LOG.warning("ChatGroq reply generation failed: %s", e, exc_info=True)
    return (
        "Thanks for contacting ShopWave. We've reviewed your case and will follow up with next steps."
    )


def _msg_done(name: str, msgs: list) -> bool:
    return any(str(m).startswith(f"{name}:") for m in (msgs or []))


def _blob(msgs: list) -> str:
    return "".join(map(str, msgs or []))


def _resolve_order_id(state: dict) -> str:
    """Prefer state.order_id; else parse ORD-* from ticket or tool log; else latest order after get_customer."""
    oid = (state.get("order_id") or "").strip().upper()
    if oid:
        return oid
    ctx = state.get("ticket", "") or ""
    msgs = state.get("messages") or []
    oid = (extract_order_id(ctx) or extract_order_id(_blob(msgs)) or "").strip().upper()
    if oid:
        return oid
    email = (state.get("customer_email") or "").strip()
    if email and _msg_done("get_customer", msgs):
        lo = latest_order_id_for_email(email)
        if lo:
            return lo.strip().upper()
    return ""


def _kb_search_query(state: dict) -> str:
    ctx = (state.get("ticket") or "").strip()
    cls = state.get("classification") or {}
    head = (ctx.split("\n")[0] if ctx else "")[:240]
    parts = [head, str(cls.get("category") or ""), str(cls.get("urgency") or "")]
    q = " ".join(p for p in parts if p).strip()
    return q[:500] if q else "return refund warranty shipping policy"


def pipeline_next_step(state: dict) -> str | None:
    # classify → KB search → customer → order → product → refund path → reply
    msgs = state.get("messages") or []
    email = (state.get("customer_email") or "").strip()
    oid = _resolve_order_id(state)

    if not _msg_done("classify_ticket", msgs):
        return "classify_ticket"
    if not _msg_done("search_knowledge_base", msgs):
        return "search_knowledge_base"
    if email and not _msg_done("get_customer", msgs):
        return "get_customer"
    if not oid:
        return None
    if not _msg_done("get_order", msgs):
        return "get_order"
    if not _msg_done("get_product", msgs):
        return "get_product"
    if not _msg_done("check_refund", msgs):
        return "check_refund"
    b = _blob(msgs)
    elig = "'eligible': True" in b or '"eligible": True' in b
    if elig and not _msg_done("issue_refund", msgs):
        return "issue_refund"
    if not _msg_done("send_reply", msgs):
        return "send_reply"
    return "done"


def _llm_thought_confidence(next_action: str, ctx: str, history: str) -> tuple[str, float]:
    if llm is None or os.getenv("SHOPWAVE_OFFLINE"):
        return f"Pipeline requires `{next_action}` per policy and ticket state.", 0.86
    prompt = (
        f"You are a ShopWave L1 routing analyst. One tool will run next: {next_action}."
        f"Knowledge excerpt:{KNOWLEDGE_BASE[:2800]}"
        f"Ticket:{ctx}"
        f"Tool log so far:{history or '(none)'}"
        "Respond with EXACTLY two lines:"
        "Line 1: One short sentence: your reasoning (why this tool is appropriate now)."
        "Line 2: A single number from 0.0 to 1.0 - your confidence that this is the correct next step."
    )
    try:
        resp = llm.invoke(prompt)
        text = (getattr(resp, "content", None) or str(resp) or "").strip()
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        thought = lines[0] if lines else "Proceed with tool."
        conf = _parse_confidence_from_llm_lines(lines, default=0.75)
        return thought, conf
    except Exception as e:
        _LOG.warning("ChatGroq invoke failed in _llm_thought_confidence: %s", e, exc_info=True)
        return "LLM unavailable; using default confidence.", 0.72


def think(state: AgentState) -> AgentState:
    run_id = state.get("run_id") or str(uuid.uuid4())
    tid = state.get("ticket_id") or "UNKNOWN"
    tc = int(state.get("think_cycle") or 0) + 1
    base: AgentState = {**state, "run_id": run_id, "think_cycle": tc}
    msgs = list(base.get("messages") or [])
    actions_taken = list(base.get("actions_taken") or [])

    if base.get("escalated"):
        return {**base, "next_step": "done"}

    if pipeline_next_step(base) == "done":
        fd = base.get("final_decision") or "completed"
        return {**base, "next_step": "done", "final_decision": fd}

    if tc > 28:
        summary = f"ticket={tid}; actions={actions_taken}; reason=max_think_cycles"
        escalate(tid, summary, "medium")
        return {
            **base,
            "next_step": "done",
            "escalated": True,
            "escalation_reason": "max_think_cycles",
            "final_decision": "escalated_safety_cap",
        }

    nxt = pipeline_next_step(base)
    ctx = base.get("ticket", "")
    history = "\n".join(msgs)

    if nxt is None:
        summary = (
            f"ticket={tid}; body_preview={ctx[:400]}; actions_taken={actions_taken}; "
            "reason=missing_order_id_or_unclear_intent"
        )
        escalate(tid, summary, "high")
        append_audit_step(
            tid,
            run_id,
            {
                "phase": "think",
                "thought": "Cannot resolve order id from ticket; escalating.",
                "action": None,
                "confidence": 0.2,
                "input": {},
                "output": {"escalate": True},
            },
        )
        return {
            **base,
            "next_step": "done",
            "escalated": True,
            "escalation_reason": "missing_order_id",
            "final_decision": "escalated_missing_order_id",
        }

    if nxt == "done":
        return {**base, "next_step": "done", "final_decision": base.get("final_decision") or "closed"}

    thought, confidence = _llm_thought_confidence(nxt, ctx, history)
    append_audit_step(
        tid,
        run_id,
        {
            "phase": "think",
            "thought": thought,
            "action": nxt,
            "input": {"stage": "routing"},
            "output": {"confidence": confidence},
            "confidence": confidence,
        },
    )

    if confidence < CONFIDENCE_THRESHOLD:
        summary = f"ticket={tid}; ticket_text={ctx[:600]}; actions={actions_taken}; reason=low_confidence ({confidence:.2f})"
        escalate(tid, summary, "medium")
        return {
            **base,
            "next_step": "done",
            "thought": thought,
            "confidence": confidence,
            "escalated": True,
            "escalation_reason": "low_confidence",
            "final_decision": "escalated_low_confidence",
        }

    oid = base.get("order_id") or extract_order_id(ctx)
    if oid:
        base = {**base, "order_id": str(oid).strip().upper()}

    return {
        **base,
        "next_step": nxt,
        "thought": thought,
        "confidence": confidence,
        "audit_chain": list(base.get("audit_chain") or [])
        + [{"phase": "think", "thought": thought, "action": nxt, "confidence": confidence}],
    }


In [22]:

def _exec_tool(step: str, state: AgentState) -> tuple[dict, dict]:
    # Run tool once; returns (input_dict, output_dict).
    ctx = state.get("ticket", "")
    oid = (state.get("order_id") or "").strip().upper() or (extract_order_id(ctx) or "")
    tid = state.get("ticket_id") or "UNKNOWN"
    email = (state.get("customer_email") or "").strip()

    if step == "classify_ticket":
        inp = {"ticket_excerpt": ctx[:1200], "customer_email": email}
        out = classify_ticket_text(ctx, email, ticket_tier=state.get("ticket_tier"))
        return inp, out
    if step == "search_knowledge_base":
        q = _kb_search_query(state)
        inp = {"query": q, "top_k": 6}
        out = search_knowledge_base(q, top_k=6)
        return inp, out
    if step == "get_customer":
        inp = {"email": email}
        out = get_customer(email=email) if email else {"found": False, "error": "no email"}
        return inp, out
    if step == "get_order":
        inp = {"order_id": oid}
        out = get_order(oid) if oid else {"error": "missing order_id"}
        return inp, out
    if step == "get_product":
        pid = (state.get("product_id") or "").strip().upper()
        if not pid and oid in ORDERS:
            pid = ORDERS[oid]["product_id"]
        inp = {"product_id": pid}
        out = get_product(pid) if pid else {"error": "missing product_id"}
        return inp, out
    if step == "check_refund":
        as_of = as_of_date_from_state(state)
        inp = {"order_id": oid, "as_of": str(as_of)}
        out = check_refund_eligibility(oid, as_of=as_of) if oid else {"eligible": False, "reason": "missing order_id"}
        return inp, out
    if step == "issue_refund":
        amt = float(ORDERS[oid]["amount"]) if oid in ORDERS else 0.0
        inp = {"order_id": oid, "amount": amt}
        out = issue_refund(oid, amt) if oid else {"status": "error", "detail": "missing order_id"}
        return inp, out
    if step == "send_reply":
        body = (state.get("reply_draft") or "").strip() or _generate_reply_message(state)
        inp = {"ticket_id": tid, "message": body[:2000]}
        out = send_reply(tid, body)
        return inp, out
    raise ValueError(f"unknown tool {step}")


def act(state: AgentState) -> AgentState:
    step = state.get("next_step") or "done"
    run_id = state.get("run_id") or str(uuid.uuid4())
    tid = state.get("ticket_id") or "UNKNOWN"
    msgs = list(state.get("messages") or [])
    actions_taken = list(state.get("actions_taken") or [])

    if step == "done":
        return {**state, "run_id": run_id, "next_step": "done", "messages": msgs, "actions_taken": actions_taken}

    inp: dict = {}
    try:

        def _once():
            nonlocal inp
            i, o = _exec_tool(step, state)
            inp = i
            return o

        out = run_with_retries(step, _once)
    except ToolRetriesExhausted as e:
        summary = (
            f"ticket={tid}; actions={actions_taken + [step]}; reason=tool_failure_after_retries: {repr(e.cause)}"
        )
        escalate(tid, summary, "high")
        try:
            inp, _ = _exec_tool(step, state)
        except Exception:
            inp = {"tool": step}
        append_audit_step(
            tid,
            run_id,
            {
                "phase": "act",
                "thought": state.get("thought", ""),
                "action": step,
                "input": inp,
                "output": {"error": str(e.cause)},
                "tool_error": True,
            },
        )
        return {
            **state,
            "run_id": run_id,
            "messages": msgs,
            "next_step": "done",
            "escalated": True,
            "tool_failure_escalation": True,
            "escalation_reason": "tool_failure",
            "final_decision": "escalated_tool_failure",
            "actions_taken": actions_taken,
        }

    append_audit_step(
        tid,
        run_id,
        {
            "phase": "act",
            "thought": state.get("thought", ""),
            "action": step,
            "input": inp,
            "output": out,
        },
    )

    actions_taken = actions_taken + [step]
    line = f"{step}: {out}"
    oid_persist = (state.get("order_id") or "").strip().upper() or extract_order_id(state.get("ticket", "") or "") or ""
    if not oid_persist and step == "get_customer" and isinstance(out, dict) and out.get("found"):
        em = (state.get("customer_email") or "").strip()
        if em:
            oid_persist = (latest_order_id_for_email(em) or "").strip().upper()
    if not oid_persist and step == "get_order" and isinstance(out, dict) and out.get("order_id"):
        oid_persist = str(out["order_id"]).strip().upper()
    new_state: AgentState = {
        **state,
        "run_id": run_id,
        "messages": msgs + [line],
        "actions_taken": actions_taken,
        "audit_chain": list(state.get("audit_chain") or []),
    }
    if oid_persist:
        new_state["order_id"] = oid_persist
    if step == "classify_ticket" and isinstance(out, dict):
        new_state["classification"] = out
    if step == "get_order" and isinstance(out, dict) and out.get("product_id"):
        new_state["product_id"] = str(out["product_id"]).strip().upper()

    if step == "send_reply":
        new_state["final_decision"] = "replied_and_closed"
    return new_state


def should_continue(state: AgentState):
    if state.get("next_step") == "done":
        return "end"
    return "continue"


In [23]:

from langgraph.graph import END, START, StateGraph

builder = StateGraph(AgentState)
builder.add_node("think", think)
builder.add_node("act", act)
builder.add_edge(START, "think")
builder.add_conditional_edges("think", should_continue, {"continue": "act", "end": END})
builder.add_edge("act", "think")
graph = builder.compile()


## 7. Concurrency — parallel tickets

`ThreadPoolExecutor`; audit writes serialized with a lock. Per-ticket JSON under `audit_logs/`.
Use `run_all_tickets()` to process every ticket in `data/tickets.json` (optionally `parallel=True`).


In [24]:

def run_one_ticket(ticket_id: str) -> dict:
    run_id = str(uuid.uuid4())
    tc = ticket_context(ticket_id)
    pre_oid = (extract_order_id(tc.get("ticket", "") or "") or "").strip().upper()
    init = {
        **tc,
        "run_id": run_id,
        "order_id": pre_oid,
        "product_id": "",
        "classification": {},
        "ticket_tier": tc.get("tier"),
        "messages": [],
        "next_step": "",
        "result": "",
        "actions_taken": [],
        "audit_chain": [],
        "escalated": False,
    }
    out = graph.invoke(init, config={"recursion_limit": 64})
    esc = bool(out.get("escalated"))
    fd = out.get("final_decision") or ("escalated" if esc else "unknown")
    actions = list(out.get("actions_taken") or [])
    finalize_ticket_audit(
        ticket_id,
        run_id,
        final_decision=fd,
        actions_taken=actions,
        escalated=esc,
        extra={"confidence_last": out.get("confidence"), "thought_last": out.get("thought")},
    )
    return {
        "ticket_id": ticket_id,
        "run_id": run_id,
        "final_decision": fd,
        "actions_taken": actions,
        "escalated": esc,
        "escalation_reason": out.get("escalation_reason"),
    }


def run_tickets_parallel(ticket_ids: list[str], *, max_workers: int = 4) -> list[dict]:
    results: list[dict] = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(run_one_ticket, tid): tid for tid in ticket_ids}
        for fut in as_completed(futs):
            results.append(fut.result())
    return sorted(results, key=lambda r: r["ticket_id"])


def _ticket_id_sort_key(tid: str) -> int:
    m = re.search(r"(\d+)$", str(tid))
    return int(m.group(1)) if m else 0


def all_ticket_ids() -> list[str]:
    return sorted(TICKETS.keys(), key=_ticket_id_sort_key)


def run_all_tickets(*, parallel: bool = False, max_workers: int = 4) -> list[dict]:
    """Run agent for every ticket in `data/tickets.json`; each run writes `audit_logs/<ticket_id>.json`."""
    ids = all_ticket_ids()
    if parallel:
        return run_tickets_parallel(ids, max_workers=max_workers)
    return [run_one_ticket(tid) for tid in ids]


In [25]:

import json as _json

# --- Single ticket ---
demo = run_one_ticket("TKT-001")
print(_json.dumps(demo, indent=2))
print("Audit file:", audit_path_for_ticket(demo["ticket_id"]))


{
  "ticket_id": "TKT-001",
  "run_id": "f12c3c7e-6da5-40b3-941e-dde7d9ab5d08",
  "final_decision": "replied_and_closed",
  "actions_taken": [
    "classify_ticket",
    "search_knowledge_base",
    "get_customer",
    "get_order",
    "get_product",
    "check_refund",
    "issue_refund",
    "send_reply"
  ],
  "escalated": false,
  "escalation_reason": null
}
Audit file: /home/pankaj/practice/Hackthon/hackathon2026-PankajPandey/audit_logs/TKT-001.json


In [26]:

# --- All tickets from data/tickets.json (one audit JSON per ticket id under audit_logs/) ---
all_results = run_all_tickets(parallel=True, max_workers=6)
print(len(all_results), "tickets processed; audit dir:", AUDIT_LOG_DIR)
print(json.dumps(all_results, indent=2))

# Smaller sample only:
# sample = run_tickets_parallel(["TKT-001", "TKT-002", "TKT-003"], max_workers=3)
# Sequential all (no thread pool): all_results = run_all_tickets(parallel=False)


20 tickets processed; audit dir: /home/pankaj/practice/Hackthon/hackathon2026-PankajPandey/audit_logs
[
  {
    "ticket_id": "TKT-001",
    "run_id": "e90fed75-b319-43bd-ab81-b64ff88a64b0",
    "final_decision": "replied_and_closed",
    "actions_taken": [
      "classify_ticket",
      "search_knowledge_base",
      "get_customer",
      "get_order",
      "get_product",
      "check_refund",
      "issue_refund",
      "send_reply"
    ],
    "escalated": false,
    "escalation_reason": null
  },
  {
    "ticket_id": "TKT-002",
    "run_id": "41f63a5e-59eb-4b65-b013-d68126dc6a9c",
    "final_decision": "replied_and_closed",
    "actions_taken": [
      "classify_ticket",
      "search_knowledge_base",
      "get_customer",
      "get_order",
      "get_product",
      "check_refund",
      "send_reply"
    ],
    "escalated": false,
    "escalation_reason": null
  },
  {
    "ticket_id": "TKT-003",
    "run_id": "ab197781-c800-4f8b-864a-7dbce895b18d",
    "final_decision": "replied_a